# Mengapa Harus Dilakukan Dekomposisi (Khususnya Cholesky) pada Gaussian Process?

Dalam formulasi analitik *Gaussian Process Regression* (GPR), kita sering melihat notasi inversi matriks dan determinan:
$$\boldsymbol{\alpha} = K_y^{-1} \mathbf{y} \quad \text{dan} \quad \log |K_y|$$

Secara simbolik di atas kertas, menuliskan $K_y^{-1}$ tampak sangat sederhana. Namun, dalam komputasi nyata (baik implementasi di pustaka seperti PyTorch, GPyTorch, GPflow, Scikit-Learn, maupun kalkulasi numerik manual), **kita hampir tidak pernah menghitung inversi langsung $K_y^{-1}$**. Kita **wajib melakukan dekomposisi**, khususnya **Dekomposisi Cholesky** ($K_y = L L^T$), karena 4 alasan fundamental berikut:

---

## 1. Kestabilan Numerik (*Numerical Stability & Condition Number*)
Matriks kovariansi kernel $K_y$ dibentuk dari fungsi korelasi spasial berbasis jarak (misalnya kernel RBF: $k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2l^2}\right)$).
* Apabila terdapat dua atau lebih titik data training yang lokasinya berdekatan di ruang input, baris dan kolom yang bersesuaian pada matriks $K_y$ akan menjadi sangat mirip (*nearly collinear*).
* Hal ini membuat matriks $K_y$ memiliki **angka kondisi (*condition number*) yang sangat tinggi/buruk** (*ill-conditioned*).
* Jika kita memaksakan inversi langsung (misalnya dengan metode eliminasi Gauss biasa atau Gauss-Jordan), galat pembulatan komputer (*floating-point roundoff error*) akan teramplifikasi secara drastis (*catastrophic numerical explosion*). Akibatnya, inversi bisa gagal atau menghasilkan vektor bobot $\boldsymbol{\alpha}$ yang melenceng jauh dari nilai sebenarnya.
* **Keunggulan Cholesky**: Karena matriks kovariansi $K_y$ bersifat simetris dan positif definit, Dekomposisi Cholesky $K_y = L L^T$ terbukti secara teoretis **sangat stabil tanpa memerlukan pivoting**, sehingga meminimalkan akumulasi galat pembulatan.

---

## 2. Mengubah Inversi Menjadi Substitusi Linier Segitiga yang Cepat & Presisi
Tujuan utama inferensi GPR sebenarnya **bukan mencari matriks inversi $K_y^{-1}$**, melainkan mencari vektor solusi dari sistem persamaan linier:
$$K_y \boldsymbol{\alpha} = \mathbf{y}$$

Dengan mendekomposisi $K_y = L L^T$ di mana $L$ adalah matriks segitiga bawah (*lower triangular matrix*):
$$L \underbrace{(L^T \boldsymbol{\alpha})}_{\mathbf{v}} = \mathbf{y}$$

Masalah inversi yang rumit dipecah menjadi dua tahap substitusi segitiga yang sangat mudah dan presisi:
1. **Forward Substitution** ($L \mathbf{v} = \mathbf{y}$):
   Karena $L$ berbentuk segitiga bawah ($L_{ij} = 0$ untuk $j > i$), elemen pertama langsung diperoleh:
   $$v_1 = \frac{y_1}{L_{11}}$$
   kemudian disubstitusikan ke baris berikutnya untuk mencari $v_2, v_3, \dots, v_N$ secara bertahap.
2. **Backward Substitution** ($L^T \boldsymbol{\alpha} = \mathbf{v}$):
   Karena $L^T$ berbentuk segitiga atas, elemen terakhir langsung diperoleh:
   $$\alpha_N = \frac{v_N}{L_{NN}}$$
   kemudian disubstitusikan mundur ke atas untuk mencari $\alpha_{N-1}, \dots, \alpha_1$.

Kedua tahapan substitusi ini hanya berorde $\mathcal{O}(N^2)$ dan sama sekali tidak melibatkan pembagian determinan matriks besar yang rawan menghasilkan kesalahan numerik.

---

## 3. Menghitung Determinan $\log |K_y|$ Tanpa Risiko *Underflow/Overflow*
Pada tahap optimasi hyperparameter (mengevaluasi *Marginal Likelihood*), kita wajib menghitung determinan matriks kovariansi:
$$\text{Complexity Penalty Term} = -\frac{1}{2} \log |K_y|$$

* Jika $|K_y|$ dihitung secara naif (misalnya lewat ekspansi kofaktor atau perkalian nilai eigen), nilai determinan dari matriks berdimensi sedang hingga besar (misal $N \ge 100$) akan dengan mudah:
  * Menjadi terlampau besar $\to$ mengalami **overflow** (menjadi `+Inf` di memori komputer).
  * Menjadi sangat mendekati nol $\to$ mengalami **underflow** (menjadi `0.0`, yang menyebabkan $\log(0) = -\infty$).
* **Solusi dengan Cholesky**:
  Karena $K_y = L L^T$, determinannya adalah perkalian determinan dari dua matriks segitiga:
  $$|K_y| = |L| \cdot |L^T| = |L|^2$$
  Determinan dari matriks segitiga $L$ hanyalah hasil kali elemen-elemen diagonalnya ($|L| = \prod_{i=1}^N L_{ii}$). Menggunakan sifat logaritma, perkalian raksasa tersebut berubah menjadi **penjumlahan skalar logaritma**:
  $$\log |K_y| = \log \left( \prod_{i=1}^N L_{ii}^2 \right) = 2 \sum_{i=1}^N \log L_{ii}$$
  Operasi ini **100% aman dan kebal terhadap bahaya numerical overflow maupun underflow**.

---

## 4. Efisiensi Komputasi (Menghemat Beban Hitung hingga 50%)
* Inversi matriks umum (seperti eliminasi Gauss atau dekomposisi LU) membutuhkan sekitar **$\frac{2}{3} N^3$ operasi titik kambang (*flops*)**.
* Dekomposisi Cholesky memanfaatkan fakta bahwa $K_y$ simetris ($K_y = K_y^T$). Algoritma hanya perlu menghitung elemen segitiga bawah dan diagonalnya saja, sehingga hanya memerlukan **$\frac{1}{3} N^3$ flops**.
* Artinya, dekomposisi Cholesky **2 kali lebih cepat** dan jauh lebih hemat memori dibandingkan metode inversi umum.

---

## 💡 Rangkuman Jawaban Singkat untuk Dosen Pembimbing
Jika saat bimbingan dosen bertanya: *"Kenapa harus didekomposisi, kenapa tidak langsung di-invers saja matriks kovariansinya?"*, Anda dapat menjawab:
1. **Stabilitas Numerik**: Matriks kovariansi kernel sering kali *ill-conditioned* (baris-barisnya hampir linear dependen akibat titik data yang berdekatan). Inversi langsung akan meledakkan galat pembulatan komputer, sedangkan Cholesky sangat stabil.
2. **Kebutuhan Sistem Linear**: Kita hanya memerlukan vektor bobot $\boldsymbol{\alpha}$, yang jauh lebih akurat dan murah diselesaikan via forward & backward substitution ($L\mathbf{v} = \mathbf{y}$ dan $L^T\boldsymbol{\alpha} = \mathbf{v}$) dibanding mencari inversi eksplisit.
3. **Mencegah Overflow/Underflow pada Determinan**: Determinan $|K_y|$ untuk *marginal likelihood* dapat dihitung secara aman lewat jumlahan logaritma diagonal: $2 \sum \log L_{ii}$.
4. **Efisiensi**: Memangkas kompleksitas komputasi menjadi separuhnya ($\frac{1}{3} N^3$ vs $\frac{2}{3} N^3$).

# Q&A Lanjutan: Apakah Dekomposisi Hanya Karena Menggunakan Komputer? Jika di Atas Kertas, Apakah Bisa Langsung Diinverskan?

### **Pertanyaan:**
> *"Jadi, dekomposisi digunakan karena kita menggunakan komputer untuk menghitung? Apakah jika menghitungnya di atas kertas maka bisa langsung di inverskan saja?"*

### **Jawaban Singkat:**
**Bisa, TAPI hanya praktis untuk ukuran yang sangat kecil ($N = 2$). Begitu ukuran data $N \ge 3$, menghitung langsung invers di atas kertas justru jauh lebih rumit dan melelahkan dibanding dekomposisi.**

---

## 1. Jika Dihitung di Atas Kertas (Aljabar Simbolik Eksak)
Di atas kertas, manusia berpikir dengan angka eksak/pecahan (misal $\frac{1}{3}$, $\sqrt{2}$, atau $e^{-0.5}$).
* Masalah **ketidakstabilan pembulatan numerik (*floating point error*)** tidak terjadi karena kita tidak membulatkan angka ke biner 64-bit seperti CPU komputer.

### Kasus $N = 2$ (Dua Titik Data):
Jika matriksnya $2 \times 2$:
$$K_y = \begin{bmatrix} a & b \\ b & c \end{bmatrix}$$
Di atas kertas, Anda **sangat mudah dan dianjurkan** langsung memakai rumus inversi adjoin-determinan:
$$K_y^{-1} = \frac{1}{ac - b^2} \begin{bmatrix} c & -b \\ -b & a \end{bmatrix}$$
Lalu tinggal kalikan $K_y^{-1} \mathbf{y}$. Pada kasus $2 \times 2$, melakukan Cholesky di atas kertas justru terasa panjang dan bertele-tele.

---

## 2. Mengapa untuk $N \ge 3$ di Atas Kertas pun Dekomposisi Tetap Jauh Lebih Nyaman?
Mari bandingkan apa yang terjadi jika Anda menghitung matriks $3 \times 3$ di atas kertas:

### Cara A: Inversi Langsung via Kofaktor / Adjoin ($K_y^{-1} = \frac{1}{|K_y|} \operatorname{Adj}(K_y)$)
1. Anda harus menghitung **determinan utama $3 \times 3$** (lewat metode Sarrus atau ekspansi baris).
2. Anda harus menghitung **9 determinan kofaktor berukuran $2 \times 2$** satu per satu untuk mengisi tiap elemen matriks adjoin.
3. Seluruh elemen hasil inversi akan berupa angka pecahan rumit dengan penyebut determinan utama.
4. Setelah itu, Anda masih harus melakukan perkalian matriks penuh $3 \times 3$ dengan vektor $\mathbf{y}$ berukuran $3 \times 1$.
$\to$ **Sangat rawan salah hitung tanda minus dan aritmatika kofaktor yang panjang.**

### Cara B: Dekomposisi Cholesky ($K_y = L L^T$)
1. Anda hanya mencari **6 angka** untuk matriks segitiga bawah $L$ (elemen di atas diagonal nol mutlak, tidak perlu dicari).
2. Anda **tidak pernah menghitung 9 kofaktor matriks**.
3. Saat mencari $\boldsymbol{\alpha}$, Anda melakukan **substitusi maju ($L \mathbf{v} = \mathbf{y}$)**:
   * Baris 1 langsung ketemu $v_1 = \frac{y_1}{L_{11}}$ (hanya satu pembagian biasa).
   * Baris 2 langsung ketemu $v_2$.
   * Baris 3 langsung ketemu $v_3$.
4. Lalu mundur di $L^T \boldsymbol{\alpha} = \mathbf{v}$ untuk mendapatkan $\alpha_3, \alpha_2, \alpha_1$.
$\to$ **Jauh lebih sedikit langkah tulisannya, lebih terstruktur, dan mudah ditiru di papan tulis.**

---

## 3. Kesimpulan Dua Perspektif

| Aspek Peninjauan | Mengapa Tidak Inversi Langsung? | Mengapa Memilih Dekomposisi? |
|---|---|---|
| **Perspektif Komputer (Numerik)** | **Kestabilan angka**: Inversi langsung pada matriks kernel yang *ill-conditioned* memicu ledakan galat pembulatan desimal (*roundoff explosion*) dan *overflow/underflow* pada determinan. | Algoritma Cholesky stabil tanpa pivoting, cepat, dan aman dari *overflow* via $2 \sum \log L_{ii}$. |
| **Perspektif Kertas (Manusia)** | **Beban aljabar**: Untuk $N \ge 3$, mencari matriks inversi eksplisit mengharuskan kita menghitung banyak sekali kofaktor minor yang sangat melelahkan. | Kita tidak butuh matriks inversnya, hanya butuh solusinya. Substitusi segitiga maju-mundur jauh lebih mudah dikerjakan manual. |

> **Fakta Sejarah**: Tokoh matematika **André-Louis Cholesky** menemukan metode dekomposisi ini pada awal abad ke-20 untuk keperluan survei topografi militer—**jauh sebelum komputer digital modern diciptakan**—justru karena ia merasa memecahkan sistem persamaan linier di atas kertas dengan faktorisasi segitiga jauh lebih cepat dan minim kesalahan dibandingkan menginvers matriks secara manual!

# Penjelasan Lengkap Aljabar: Bagaimana Dekomposisi Cholesky Menghasilkan Parameter Distribusi Normal Posterior?

Di dalam teori *Gaussian Process Regression* (GPR), distribusi posterior untuk titik uji $X_*$ berbentuk distribusi normal multivariat:
$$\mathbf{f}_* \mid X, \mathbf{y}, X_* \sim \mathcal{N}\big(\bar{\mathbf{f}}_*, \operatorname{cov}(\mathbf{f}_*)\big)$$

Dua parameter yang wajib kita hitung secara aljabar adalah:
1. **Parameter Mean Posterior**: $\bar{\mathbf{f}}_* = K(X_*, X) K_y^{-1} \mathbf{y} \in \mathbb{R}^{N_* \times 1}$
2. **Parameter Kovariansi/Variansi Posterior**: $\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - K(X_*, X) K_y^{-1} K(X, X_*) \in \mathbb{R}^{N_* \times N_*}$

Keduanya sama-sama memuat suku inversi $K_y^{-1}$. Mari kita bedah langkah aljabar penurunan dekomposisi $K_y = L L^T$ untuk menghitung kedua parameter tersebut tanpa pernah mencari $K_y^{-1}$ secara eksplisit.

---

## 1. Aljabar Penurunan Parameter Mean Posterior ($\bar{\mathbf{f}}_*$)

Rumus analitik mean adalah:
$$\bar{\mathbf{f}}_* = K(X_*, X) \underbrace{\left[ K_y^{-1} \mathbf{y} \right]}_{\boldsymbol{\alpha}}$$

### Langkah A: Definisikan Vektor Bobot $\boldsymbol{\alpha}$
Kita ingin mencari vektor $\boldsymbol{\alpha} \in \mathbb{R}^{N \times 1}$ sedemikian sehingga:
$$\boldsymbol{\alpha} = K_y^{-1} \mathbf{y} \iff K_y \boldsymbol{\alpha} = \mathbf{y}$$

### Langkah B: Substitusikan Faktorisasi Cholesky $K_y = L L^T$
$$(L L^T) \boldsymbol{\alpha} = \mathbf{y} \implies L \left( L^T \boldsymbol{\alpha} \right) = \mathbf{y}$$

### Langkah C: Pecah Menjadi Dua Sistem Linier Segitiga
Definisikan vektor perantara $\mathbf{v} = L^T \boldsymbol{\alpha} \in \mathbb{R}^{N \times 1}$. Persamaan menjadi sistem berantai:
$$\begin{cases}
1.\quad L \mathbf{v} = \mathbf{y} & \text{(Sistem Segitiga Bawah / Forward Substitution)} \\[0.8em]
2.\quad L^T \boldsymbol{\alpha} = \mathbf{v} & \text{(Sistem Segitiga Atas / Backward Substitution)}
\end{cases}$$

Secara aljabar elemen per elemen:
* **Menyelesaikan $L \mathbf{v} = \mathbf{y}$ (Maju dari $i = 1$ hingga $N$):**
  $$\begin{bmatrix} 
  L_{11} & 0 & \dots & 0 \\ 
  L_{21} & L_{22} & \dots & 0 \\ 
  \vdots & \vdots & \ddots & \vdots \\ 
  L_{N1} & L_{N2} & \dots & L_{NN} 
  \end{bmatrix} 
  \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \end{bmatrix} 
  = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_N \end{bmatrix}$$
  * Baris 1: $L_{11} v_1 = y_1 \implies v_1 = \frac{y_1}{L_{11}}$
  * Baris 2: $L_{21} v_1 + L_{22} v_2 = y_2 \implies v_2 = \frac{y_2 - L_{21} v_1}{L_{22}}$
  * Rumus umum rekursif: $$v_i = \frac{1}{L_{ii}} \left( y_i - \sum_{k=1}^{i-1} L_{ik} v_k \right)$$

* **Menyelesaikan $L^T \boldsymbol{\alpha} = \mathbf{v}$ (Mundur dari $i = N$ hingga $1$):**
  $$\begin{bmatrix} 
  L_{11} & L_{21} & \dots & L_{N1} \\ 
  0 & L_{22} & \dots & L_{N2} \\ 
  \vdots & \vdots & \ddots & \vdots \\ 
  0 & 0 & \dots & L_{NN} 
  \end{bmatrix} 
  \begin{bmatrix} \alpha_1 \\ \alpha_2 \\ \vdots \\ \alpha_N \end{bmatrix} 
  = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \end{bmatrix}$$
  * Baris $N$: $L_{NN} \alpha_N = v_N \implies \alpha_N = \frac{v_N}{L_{NN}}$
  * Baris $N-1$: $L_{N-1,N-1} \alpha_{N-1} + L_{N, N-1} \alpha_N = v_{N-1} \implies \alpha_{N-1} = \frac{v_{N-1} - L_{N, N-1} \alpha_N}{L_{N-1, N-1}}$
  * Rumus umum mundur: $$\alpha_i = \frac{1}{L_{ii}} \left( v_i - \sum_{k=i+1}^N L_{ki} \alpha_k \right)$$

### Langkah D: Hitung Mean Prediktif Akhir
Setelah vektor $\boldsymbol{\alpha}$ diperoleh utuh, kalikan dengan matriks kovariansi silang $K(X_*, X)$:
$$\bar{\mathbf{f}}_* = K(X_*, X) \boldsymbol{\alpha} = \sum_{i=1}^N \alpha_i K(X_*, \mathbf{x}_i)$$
*(Selesai! Parameter mean posterior diperoleh murni lewat eliminasi aljabar biasa).* 

---

## 2. Aljabar Penurunan Parameter Kovariansi Posterior ($\operatorname{cov}(\mathbf{f}_*)$)

Rumus analitik kovariansi posterior adalah:
$$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - \underbrace{K(X_*, X) K_y^{-1} K(X, X_*)}_{\Omega}$$
Kita perlu menghitung suku pengurang $\Omega \in \mathbb{R}^{N_* \times N_*}$ secara cerdas menggunakan dekomposisi Cholesky $K_y = L L^T$.

### Langkah A: Manipulasi Aljabar Menggunakan Sifat Transpose & Invers
Ingat sifat dasar aljabar invers perkalian matriks:
$$(A B)^{-1} = B^{-1} A^{-1} \implies (L L^T)^{-1} = (L^T)^{-1} L^{-1} = L^{-T} L^{-1}$$

Substitusikan ke dalam suku $\Omega$:
$$\Omega = K(X_*, X) \left[ L^{-T} L^{-1} \right] K(X, X_*)$$

Ingat bahwa matriks kovariansi silang bersifat transpose: $K(X_*, X) = K(X, X_*)^T$. Maka:
$$\Omega = K(X, X_*)^T L^{-T} L^{-1} K(X, X_*)$$

Dengan sifat aljabar transpose $(A B)^T = B^T A^T$, perhatikan suku:
$$K(X, X_*)^T L^{-T} = \left( L^{-1} K(X, X_*) \right)^T$$

### Langkah B: Definisikan Matriks Perantara $W$
Definisikan matriks $W \in \mathbb{R}^{N \times N_*}$ sebagai:
$$W = L^{-1} K(X, X_*)$$

Maka secara otomatis:
$$W^T = \left( L^{-1} K(X, X_*) \right)^T = K(X, X_*)^T L^{-T} = K(X_*, X) L^{-T}$$

Dengan demikian, suku pengurang $\Omega$ yang tadinya sangat rumit **berubah menjadi bentuk perkalian matriks simetris yang sangat elegan**:
$$\Omega = W^T W$$

### Langkah C: Cara Mencari Matriks $W$ Kolom Demi Kolom Tanpa Menginvers $L$
*(Bagian ini yang menjadi fokus pertanyaan Anda: Bagaimana dari $W = L^{-1} K(X, X_*)$ bisa berubah menjadi $L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$?)*

Mari kita bedah struktur internal matriks $W$ dan matriks $K(X, X_*)$ kolom per kolom!

1. **Tinjau Struktur Matriks $K(X, X_*)$ Berukuran $N \times N_*$**:
   Matriks kovariansi silang ini menghubungkan $N$ data training $X = [\mathbf{x}_1, \dots, \mathbf{x}_N]^T$ dengan $N_*$ titik uji $X_* = [\mathbf{x}_{*1}, \dots, \mathbf{x}_{*N_*}]^T$.
   Matriks ini tersusun atas kolom-kolom vektor:
   $$K(X, X_*) = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix} \in \mathbb{R}^{N \times N_*}$$
   di mana kolom ke-$j$ adalah vektor kovariansi titik uji ke-$j$ terhadap seluruh data training:
   $$\mathbf{k}_{*j} = K(X, \mathbf{x}_{*j}) = \begin{bmatrix} k(\mathbf{x}_1, \mathbf{x}_{*j}) \\ k(\mathbf{x}_2, \mathbf{x}_{*j}) \\ \vdots \\ k(\mathbf{x}_N, \mathbf{x}_{*j}) \end{bmatrix} \in \mathbb{R}^{N \times 1}$$

2. **Tinjau Struktur Matriks $W$ Berukuran $N \times N_*$**:
   Sama halnya dengan $K(X, X_*)$, matriks $W$ juga tersusun atas $N_*$ buah vektor kolom:
   $$W = \begin{bmatrix} \mathbf{w}_1 & \mathbf{w}_2 & \dots & \mathbf{w}_{N_*} \end{bmatrix} \in \mathbb{R}^{N \times N_*}$$
   di mana masing-masing kolom $\mathbf{w}_j = [w_{1j}, w_{2j}, \dots, w_{Nj}]^T \in \mathbb{R}^{N \times 1}$.

3. **Kalikan Persamaan $W = L^{-1} K(X, X_*)$ dengan $L$ dari Kiri**:
   $$L W = L \left( L^{-1} K(X, X_*) \right) \implies L W = K(X, X_*)$$

4. **Tuliskan dalam Bentuk Blok Kolom**:
   Sesuai sifat dasar aljabar linier, perkalian matriks dengan matriks terbagi menjadi perkalian matriks dengan masing-masing kolomnya:
   $$L \begin{bmatrix} \mathbf{w}_1 & \mathbf{w}_2 & \dots & \mathbf{w}_{N_*} \end{bmatrix} = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix}$$
   $$\begin{bmatrix} L \mathbf{w}_1 & L \mathbf{w}_2 & \dots & L \mathbf{w}_{N_*} \end{bmatrix} = \begin{bmatrix} \mathbf{k}_{*1} & \mathbf{k}_{*2} & \dots & \mathbf{k}_{*N_*} \end{bmatrix}$$

5. **Pemisahan Sistem Linier per Kolom**:
   Dengan menyamakan kolom ke-$j$ di sisi kiri dan sisi kanan, kita peroleh sistem persamaan linier independen untuk tiap titik uji ke-$j$ ($j = 1, \dots, N_*$):
   $$L \mathbf{w}_j = \mathbf{k}_{*j} \iff L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$$

   **Mengapa bentuk ini sangat menguntungkan?**
   Karena $L$ adalah matriks segitiga bawah (*lower triangular*), persamaan $L \mathbf{w}_j = K(X, \mathbf{x}_{*j})$ diselesaikan dengan **Forward Substitution biasa** dari baris 1 sampai baris $N$:
   $$\begin{bmatrix} 
   L_{11} & 0 & 0 \\ 
   L_{21} & L_{22} & 0 \\ 
   L_{31} & L_{32} & L_{33} 
   \end{bmatrix} 
   \begin{bmatrix} w_{1j} \\ w_{2j} \\ w_{3j} \end{bmatrix}
   = \begin{bmatrix} k(\mathbf{x}_1, \mathbf{x}_{*j}) \\ k(\mathbf{x}_2, \mathbf{x}_{*j}) \\ k(\mathbf{x}_3, \mathbf{x}_{*j}) \end{bmatrix}$$
   * $w_{1j} = \frac{k(\mathbf{x}_1, \mathbf{x}_{*j})}{L_{11}}$
   * $w_{2j} = \frac{k(\mathbf{x}_2, \mathbf{x}_{*j}) - L_{21} w_{1j}}{L_{22}}$
   * $w_{3j} = \frac{k(\mathbf{x}_3, \mathbf{x}_{*j}) - L_{31} w_{1j} - L_{32} w_{2j}}{L_{33}}$
   *(Sangat mudah dan cepat, persis sama seperti mencari $\mathbf{v}$ pada perhitungan mean sebelumnya!).*

### Langkah D: Hitung Kovariansi & Variansi Posterior Akhir
Setelah semua kolom $\mathbf{w}_j$ ditemukan sehingga matriks $W = [\mathbf{w}_1, \dots, \mathbf{w}_{N_*}]$ lengkap:
$$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - W^T W$$

Perhatikan elemen matriks $W^T W$:
* Elemen diagonal ke-$j$ (variansi titik uji ke-$j$):
  $$[W^T W]_{jj} = \mathbf{w}_j^T \mathbf{w}_j = \|\mathbf{w}_j\|^2 = \sum_{i=1}^N w_{ij}^2$$
  Maka **variansi marjinal prediktif** pada titik $\mathbf{x}_{*j}$ adalah:
  $$\mathbb{V}[f_{*j}] = k(\mathbf{x}_{*j}, \mathbf{x}_{*j}) - \|\mathbf{w}_j\|^2$$

* Elemen non-diagonal baris $j$ kolom $k$ (kovariansi silang antara titik uji $\mathbf{x}_{*j}$ dan $\mathbf{x}_{*k}$):
  $$[W^T W]_{jk} = \mathbf{w}_j^T \mathbf{w}_k = \sum_{i=1}^N w_{ij} w_{ik}$$
  Maka **kovariansi posterior** antara dua titik uji adalah:
  $$\operatorname{cov}(f_{*j}, f_{*k}) = k(\mathbf{x}_{*j}, \mathbf{x}_{*k}) - \mathbf{w}_j^T \mathbf{w}_k$$

---

## 3. Rangkuman Peta Alur Aljabar Cholesky pada GPR

```
      Matriks Kovariansi Ky = K(X, X) + sigma_n^2 * I
                             │
                             ▼
                [ Faktorisasi Cholesky ]
                       Ky = L * L^T
                             │
             ┌───────────────┴───────────────┐
             ▼                               ▼
   [ MENCARI MEAN f_* ]            [ MENCARI KOVARIANSI cov(f_*) ]
             │                               │
   1. Selesaikan: L * v = y        1. Selesaikan tiap kolom j:
      (Forward substitution)          L * w_j = K(X, x_*j)
             │                        (Forward substitution)
             │                               │
   2. Selesaikan: L^T * alpha = v  2. Hitung suku pengurang:
      (Backward substitution)         Omega = W^T * W
             │                               │
   3. Kalikan:                     3. Kurangkan dari Prior:
      f_* = K(X_*, X) * alpha         cov(f_*) = K(X_*, X_*) - W^T * W
```

### Keindahan Aljabar Ini:
1. **Tidak ada satu pun proses inversi matriks** ($K_y^{-1}$ hilang digantikan oleh sifat $(L L^T)^{-1} = L^{-T} L^{-1}$). 
2. Variansi posterior $\mathbb{V}[f_{*j}] = k_{**} - \|\mathbf{w}_j\|^2$ secara aljabar menjamin nilai variansi **selalu berkurang atau sama dengan prior** ($k_{**}$ dikurangi kuadrat norma $\|\mathbf{w}_j\|^2 \ge 0$). Ini mencerminkan hukum probabilitas: *informasi data training tidak pernah menambah ketidakpastian prior*.

# Q&A: Mengapa Hanya Blok Kovariansi $K(X, X)$ yang Memiliki Komponen Galat?

Pertanyaan mendasar: Pada matriks kovariansi gabungan (*joint Gaussian prior*):
$$\begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix},
\begin{bmatrix}
K_y & K(X, X_*) \\
K(X_*, X) & K(X_*, X_*)
\end{bmatrix}
\right)$$
mengapa hanya blok kiri-atas yang bernilai $K_y = K(X,X) + \sigma_n^2 I_N$, sedangkan tiga blok lainnya ($K(X, X_*)$, $K(X_*, X)$, dan $K(X_*, X_*)$) tidak memiliki suku galat $\sigma_n^2 I$?

---

## 1. Meninjau Definisi Variabel yang Dipasangkan
Matriks kovariansi gabungan sebenarnya adalah kovariansi berpasangan antara vektor observasi training $\mathbf{y}$ dan vektor fungsi laten test $\mathbf{f}_*$:

$$\operatorname{cov}\left( \begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \right) = 
\begin{bmatrix}
\operatorname{cov}(\mathbf{y}, \mathbf{y}) & \operatorname{cov}(\mathbf{y}, \mathbf{f}_*) \\
\operatorname{cov}(\mathbf{f}_*, \mathbf{y}) & \operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*)
\end{bmatrix}$$

Ingat model observasinya:
* Data training observasi: $\mathbf{y} = \mathbf{f} + \boldsymbol{\epsilon}$, di mana $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \sigma_n^2 I_N)$.
* Nilai fungsi test yang ingin diprediksi: $\mathbf{f}_* = f(X_*)$ (secara sengaja dipilih sebagai **fungsi murni / laten**, tanpa galat).

---

## 2. Bedah Aljabar 4 Blok Kovariansi

### A. Blok Kiri-Atas: $\operatorname{cov}(\mathbf{y}, \mathbf{y}) = K(X, X) + \sigma_n^2 I_N$
Kedua variabel yang dipasangkan adalah data training yang tercemar galat pengukuran:
$$\begin{aligned}
\operatorname{cov}(\mathbf{y}, \mathbf{y}) &= \operatorname{cov}(\mathbf{f} + \boldsymbol{\epsilon}, \; \mathbf{f} + \boldsymbol{\epsilon}) \\
&= \operatorname{cov}(\mathbf{f}, \mathbf{f}) + \underbrace{\operatorname{cov}(\mathbf{f}, \boldsymbol{\epsilon})}_{0} + \underbrace{\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f})}_{0} + \operatorname{cov}(\boldsymbol{\epsilon}, \boldsymbol{\epsilon}) \\
&= K(X, X) + \sigma_n^2 I_N
\end{aligned}$$
$\implies$ Karena $\boldsymbol{\epsilon}$ bertemu dengan sesamanya, muncul variansi galat $\sigma_n^2 I_N$.

---

### B. Blok Kanan-Bawah: $\operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*) = K(X_*, X_*)$
Perhatikan notasinya: kita menggunakan $\mathbf{f}_*$, bukan $\mathbf{y}_*$.
* $\mathbf{f}_*$ adalah **fungsi laten murni** pada test points, sehingga secara matematis **noise-free**:
  $$\mathbf{f}_* = f(X_*)$$
* Karena tidak ada komponen galat $\boldsymbol{\epsilon}_*$ pada $\mathbf{f}_*$, maka kovariansinya murni berasal dari fungsi kernel:
  $$\operatorname{cov}(\mathbf{f}_*, \mathbf{f}_*) = K(X_*, X_*)$$

*(Catatan: Jika tujuan Anda adalah memprediksi nilai pengukuran sensor baru $\mathbf{y}_* = \mathbf{f}_* + \boldsymbol{\epsilon}_*$, barulah blok ini bernilai $K(X_*, X_*) + \sigma_n^2 I$).*

---

### C. Blok Silang: $\operatorname{cov}(\mathbf{y}, \mathbf{f}_*) = K(X, X_*)$
Blok ini mengukur kovariansi silang antara observasi training dengan fungsi laten test:
$$\begin{aligned}
\operatorname{cov}(\mathbf{y}, \mathbf{f}_*) &= \operatorname{cov}(\mathbf{f} + \boldsymbol{\epsilon}, \; \mathbf{f}_*) \\
&= \operatorname{cov}(\mathbf{f}, \mathbf{f}_*) + \operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*)
\end{aligned}$$
* $\operatorname{cov}(\mathbf{f}, \mathbf{f}_*) = K(X, X_*)$ (korelasi antar fungsi murni melalui fungsi kernel).
* $\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*) = 0$, karena **galat instrumen pengukuran masa lalu ($\boldsymbol{\epsilon}$) tidak mungkin berkorelasi dengan fungsi murni masa depan ($\mathbf{f}_*$)**. Keduanya saling bebas (*independent*).

Sehingga suku galat lenyap, menyisakan $K(X, X_*)$. Begitu pula untuk transpose-nya $K(X_*, X) = K(X, X_*)^T$.

---

## 3. Rangkuman Inti untuk Jawaban ke Dosen
1. **Target inferensi kita adalah fungsi murni ($\mathbf{f}_*$)**: Kita ingin merekonstruksi kurva asli yang bersih dari gangguan alat ukur (*noise-free reconstruction*). Oleh karena itu, $\mathbf{f}_*$ tidak mengandung $\boldsymbol{\epsilon}$.
2. **Independensi galat**: Galat pengukuran training $\boldsymbol{\epsilon}$ bersifat independen terhadap nilai fungsi murni di mana pun, sehingga kovariansi silangnya bernilai nol ($\operatorname{cov}(\boldsymbol{\epsilon}, \mathbf{f}_*) = 0$).
3. **Akibatnya**: Suku $\sigma_n^2 I_N$ hanya hidup di blok $\operatorname{cov}(\mathbf{y}, \mathbf{y}) = K_y$.

# Q&A: Apa Kegunaan Subbab 4.2 (Teorema Partisi Gaussian Bersyarat / Schur Complement)?

Mengapa Subbab 4.2 perlu ada di dalam dokumen dan tidak langsung saja ke rumus prediksi di Subbab 4.3?

---

## 1. Inti Masalah yang Dihadapi
Di Subbab 4.1, kita baru memiliki **distribusi bersama (*Joint Gaussian Prior*)** berukuran $(N + N_*)$:
$$\begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix},
\begin{bmatrix}
K_y & K(X, X_*) \\
K(X_*, X) & K(X_*, X_*)
\end{bmatrix}
\right)$$

Ini adalah distribusi gabungan sebelum kita memproses data. Padahal tujuan utama Machine Learning / Regresi adalah **melakukan prediksi bersyarat**:
$$\text{"Jika kita sudah mengamati data latih } \mathbf{y}\text{, bagaimanakah peluang nilai fungsi di titik uji } \mathbf{f}_*\text{?"}$$
yaitu mencari distribusi bersyarat $p(\mathbf{f}_* \mid \mathbf{y})$.

---

## 2. Peran Subbab 4.2: Menyediakan Teorema Baku Statistik
Subbab 4.2 menyediakan rumus umum aljabar linear dan probabilitas Gaussian multivariat.

Jika dua kelompok variabel acak sembarang $\mathbf{z}_A$ dan $\mathbf{z}_B$ berdistribusi normal bersama:
$$\begin{bmatrix} \mathbf{z}_A \\ \mathbf{z}_B \end{bmatrix} \sim \mathcal{N}\left(
\begin{bmatrix} \boldsymbol{\mu}_A \\ \boldsymbol{\mu}_B \end{bmatrix},
\begin{bmatrix} \Sigma_{AA} & \Sigma_{AB} \\ \Sigma_{BA} & \Sigma_{BB} \end{bmatrix}
\right)$$
maka distribusi bersyarat $\mathbf{z}_B$ jika $\mathbf{z}_A$ diketahui juga **pasti berdistribusi normal secara eksak**:
$$\mathbf{z}_B \mid \mathbf{z}_A \sim \mathcal{N}(\boldsymbol{\mu}_{B|A}, \Sigma_{B|A})$$
dengan rumus baku:
$$\begin{aligned}
\boldsymbol{\mu}_{B|A} &= \boldsymbol{\mu}_B + \Sigma_{BA} \Sigma_{AA}^{-1} (\mathbf{z}_A - \boldsymbol{\mu}_A) \\
\Sigma_{B|A} &= \Sigma_{BB} - \Sigma_{BA} \Sigma_{AA}^{-1} \Sigma_{AB}
\end{aligned}$$

---

## 3. Kegunaan di Subbab 4.3: Tinggal Mencocokkan Variabel (*Plug & Play*)
Dengan adanya Subbab 4.2, penurunan rumus di Subbab 4.3 menjadi sangat elegan dan runtut tanpa perlu menurunkan integral dari nol:

Kita tinggal memetakan variabel dokumen ke teorema Subbab 4.2:
* $\mathbf{z}_A \leftarrow \mathbf{y}$ (data training) dengan mean $\boldsymbol{\mu}_A = \mathbf{0}$
* $\mathbf{z}_B \leftarrow \mathbf{f}_*$ (fungsi laten test) dengan mean $\boldsymbol{\mu}_B = \mathbf{0}$
* $\Sigma_{AA} \leftarrow K_y = K(X,X) + \sigma_n^2 I_N$
* $\Sigma_{BA} \leftarrow K(X_*, X)$
* $\Sigma_{AB} \leftarrow K(X, X_*)$
* $\Sigma_{BB} \leftarrow K(X_*, X_*)$

Substitusikan ke rumus Subbab 4.2:
1. **Mean Prediksi Posterior:**
   $$\bar{\mathbf{f}}_* = \mathbf{0} + K(X_*, X) K_y^{-1} (\mathbf{y} - \mathbf{0}) = K(X_*, X) K_y^{-1} \mathbf{y}$$
2. **Kovariansi Posterior:**
   $$\operatorname{cov}(\mathbf{f}_*) = K(X_*, X_*) - K(X_*, X) K_y^{-1} K(X, X_*)$$

Kedua persamaan di atas adalah **jantung persamaan dari Gaussian Process Regression**. Tanpa Subbab 4.2, kedua rumus ini akan terkesan "jatuh dari langit".

---

## 4. Makna Fisis Schur Complement (Tanda Minus $- $)
Suku $\Sigma_{BB} - \Sigma_{BA}\Sigma_{AA}^{-1}\Sigma_{AB}$ disebut sebagai **Schur Complement** dari blok $\Sigma_{AA}$.

Makna fisis dari tanda minus ($-$) tersebut sangat intuitif:
* $K(X_*, X_*)$ adalah **ketidakpastian awal (*prior uncertainty*)** di titik uji sebelum kita punya data apa pun.
* Suku $K(X_*, X) K_y^{-1} K(X, X_*)$ bernilai selalu semi-definit positif.
* Oleh karena itu, pengurangan ini menunjukkan bahwa **adanya data observasi $\mathbf{y}$ selalu MENGURANGI ketidakpastian** di titik uji. Semakin dekat titik uji ke data latih, suku pengurang semakin besar, sehingga variansi posteriornya mendekati nol (sangat yakin).

---

## 5. Ringkasan Singkat untuk Dosen
> *"Subbab 4.2 berfungsi sebagai landasan teoretis fundamental (teorema conditioning Gaussian multivariat) yang menjembatani distribusi gabungan prior di Subbab 4.1 menuju formulasi analitik posterior di Subbab 4.3. Melalui Schur Complement pada Subbab 4.2, penurunan rumus mean dan variansi prediksi GPR dapat dibuktikan secara matematis dan runtut."*